In [9]:
import torch
from torch_geometric.datasets import KarateClub
import yaml

dataset = KarateClub()
print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.

val_node_indices = torch.where(data.train_mask == True)[0]+2

val_mask = torch.zeros(data.y.size(0), dtype=torch.bool)
for i in val_node_indices:
    val_mask[i.item()] = True
    
test_node_indices = torch.where((data.train_mask == False) & (val_mask == False))[0]
test_mask = torch.zeros(data.y.size(0), dtype=torch.bool)
for i in test_node_indices:
    test_mask[i.item()] = True
    
data.val_mask = val_mask
data.test_mask = test_mask

Dataset: KarateClub():
Number of graphs: 1
Number of features: 34
Number of classes: 4


In [10]:
# Load best config from wandb sweep
# path = "./wandb/sweep-49ho2gio/config-cx1mu4qd.yaml" # (GCN) put best config path here 
path = "./wandb/sweep-m8fl3x44/config-torpvxmr.yaml" # (GAT) put best config path here

# path = "./wandb/sweep-2sl6k1ms/config-eouqxlrp.yaml" # (GAT) put best config path here
# path = "./wandb/sweep-utsldekv/config-uh7mds6z.yaml" # (GraphSAGE) put best config path here
# path = "./wandb/sweep-vllbse97/config-t3yov5ho.yaml" # (DNN) put best config path here
with open(path, "r", encoding="utf-8") as f:
    raw_config = yaml.safe_load(f)

# W&B YAML stores each parameter as {value: ...}.
best_config = {
    key: entry["value"]
    for key, entry in raw_config.items()
    if isinstance(entry, dict) and "value" in entry
}

def config_kwargs(prefix):
    return {
        key[len(prefix):]: value # round(value, 3)
        for key, value in best_config.items()
        if key.startswith(prefix)
    }

In [11]:
best_config

{'models.GAT.dropout': 0.2659104860973238,
 'models.GAT.heads': 2,
 'models.GAT.hidden_channels': 32,
 'models.GAT.num_layers': 2,
 'optimizer.Adam.lr': 0.002652587179851886,
 'optimizer.Adam.weight_decay': 6.784120880898644e-05,
 'training.gradient_clip_val': 1.384623326530933,
 'wandb_project': 'karate-club-gcn-sweep'}

In [12]:
import torch
import torch.nn.functional as F

from utils.model import GCN, GAT, GraphSAGE, DNN

import random

def seed_everything(seed):
    """ Set random seeds for reproducibility.
    Args:
        seed (int): Random seed.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
# model_kwargs = config_kwargs("models.GCN.")
# model = GCN(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
model_kwargs = config_kwargs("models.GAT.")
model = GAT(in_channels=dataset.num_features,
            num_classes=dataset.num_classes,
            **model_kwargs)
# model_kwargs = config_kwargs("models.GraphSAGE.")
# model = GraphSAGE(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
# model_kwargs = config_kwargs("models.DNN.")
# model = DNN(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)

print("Model config:", model_kwargs)
print(model)

Model config: {'dropout': 0.2659104860973238, 'heads': 2, 'hidden_channels': 32, 'num_layers': 2}
GAT(
  (convs): ModuleList(
    (0): GATConv(34, 32, heads=2)
    (1): GATConv(32, 32, heads=2)
  )
  (classifier): Linear(in_features=32, out_features=4, bias=True)
)


In [13]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer_kwargs = config_kwargs("optimizer.Adam.")
# PyYAML may parse scientific notation as a string.
optimizer_kwargs = {key: float(value) for key, value in optimizer_kwargs.items()}
optimizer = torch.optim.Adam(model.parameters(), **optimizer_kwargs)
gradient_clip_val = float(best_config["training.gradient_clip_val"])
print("Adam config:", optimizer_kwargs)
print("Gradient clip:", gradient_clip_val)

def train(data):
    model.train()
    optimizer.zero_grad()  # Clear gradients.
    out = model(data.x, data.edge_index)  # Perform a single forward pass.
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
    loss.backward()  # Derive gradients.
    if gradient_clip_val > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_val)
    optimizer.step()  # Update parameters based on gradients.
    return loss

loss_history = []
for epoch in range(500):
    loss = train(data)
    loss_history.append(loss.item())
    if epoch % 10 == 0:
        print(f"epoch={epoch}, loss={loss.item():.4f}")

Adam config: {'lr': 0.002652587179851886, 'weight_decay': 6.784120880898644e-05}
Gradient clip: 1.384623326530933
epoch=0, loss=1.3817
epoch=10, loss=1.3663
epoch=20, loss=1.3279
epoch=30, loss=1.2628
epoch=40, loss=1.1310
epoch=50, loss=0.8874
epoch=60, loss=0.7574
epoch=70, loss=0.3525
epoch=80, loss=0.2951
epoch=90, loss=0.1119
epoch=100, loss=0.1671
epoch=110, loss=0.0347
epoch=120, loss=0.0205
epoch=130, loss=0.0203
epoch=140, loss=0.0049
epoch=150, loss=0.0077
epoch=160, loss=0.0142
epoch=170, loss=0.0060
epoch=180, loss=0.0178
epoch=190, loss=0.0258
epoch=200, loss=0.0025
epoch=210, loss=0.0030
epoch=220, loss=0.0174
epoch=230, loss=0.0515
epoch=240, loss=0.0060
epoch=250, loss=0.0023
epoch=260, loss=0.0029
epoch=270, loss=0.0004
epoch=280, loss=0.0141
epoch=290, loss=0.0552
epoch=300, loss=0.0002
epoch=310, loss=0.0014
epoch=320, loss=0.0002
epoch=330, loss=0.0030
epoch=340, loss=0.0043
epoch=350, loss=0.0004
epoch=360, loss=0.0010
epoch=370, loss=0.0003
epoch=380, loss=0.0011


In [ ]:
# import numpy as np

# np.save("./loss_curve_vis/data/gat_train_loss_hpt.npy", np.array(loss_history))

In [14]:
model.eval()
logits = model(data.x, data.edge_index)

test_loss = F.cross_entropy(logits[data.test_mask], data.y[data.test_mask]).item()
test_accuracy = (
        F.softmax(logits[data.test_mask],dim=1).argmax(dim=1) == data.y[data.test_mask]
    ).float().mean().item()

print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

test loss:  0.8682469129562378
test accuracy:  0.807692289352417


In [ ]:
# 예측 값
F.softmax(logits[data.test_mask],dim=1).argmax(dim=1)

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1])

In [ ]:
# 실제 값
data.y[data.test_mask]

tensor([1, 1, 1, 3, 3, 1, 1, 3, 1, 1, 1, 0, 0, 3, 1, 0, 1, 0, 1, 0, 0, 2, 0, 0,
        2, 0, 0, 2, 0, 0])